# 20 — Baseline classiche sullo split comune

Stesso protocollo che useranno i modelli quantistici: split `splits/split_files_v2.npz`, preprocessing stimato sul solo training, 8 feature standardizzate.

- **Test di riferimento** (`test`): 5.000 righe del file di test con vettori di feature **mai visti** nel file di training.
- **`test_original`**: 5.000 righe del test così com'è, solo per confronto.
- Addestramento su **5.000 righe** (come i circuiti) e su **20.000 righe**; 5 seed per i modelli stocastici.
- `MLP small (10)` ha ~101 parametri, come i circuiti quantistici; `MLP (100, 50)` è l'architettura di `prova.py`.

LightGBM e XGBoost non sono installati in questo ambiente: al loro posto c'è `HistGradientBoostingClassifier` di scikit-learn, lo stesso tipo di modello.

In [1]:
import os, sys, time, json
from pathlib import Path
REPO = Path.cwd()
while not (REPO / 'qdetect').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
import numpy as np, pandas as pd
import qdetect as qd
from qdetect import baselines as B

split = qd.load_split(REPO / 'splits/split_files_v2.npz')
P = qd.prepare(split)
RESULTS = REPO / 'results' / 'baselines_classical.csv'
display(split.summary())

,set,n,frac_malicious
0,train,5000,0.49940
1,train_large,20000,0.49945
2,val,1000,0.50500
3,test,5000,0.50000
4,test_original,5000,0.50000


## 1. Baseline su 5.000 e 20.000 righe

In [2]:
t0 = time.time()
res = B.run_baselines(P)
print(f'totale {time.time()-t0:.0f} s')
RESULTS.parent.mkdir(exist_ok=True)
res.to_csv(RESULTS, index=False)

Logistic Regression  train        seed=42    test=0.8949  test_original=0.9934  fit=0.0s


Logistic Regression  train_large  seed=42    test=0.9020  test_original=0.9939  fit=0.0s


MLP small (10)       train        seed=42    test=0.9135  test_original=0.9954  fit=0.2s


MLP small (10)       train        seed=123   test=0.8948  test_original=0.9917  fit=0.1s


MLP small (10)       train        seed=2024  test=0.9142  test_original=0.9958  fit=0.3s


MLP small (10)       train        seed=2025  test=0.9038  test_original=0.9921  fit=0.2s


... (35 outputs left out)

Gradient Boosting    train_large  seed=2024  test=0.9776  test_original=0.9993  fit=0.3s


Gradient Boosting    train_large  seed=2025  test=0.9789  test_original=0.9991  fit=0.4s


Gradient Boosting    train_large  seed=2026  test=0.9771  test_original=0.9991  fit=0.3s


totale 22 s


In [3]:
summary = B.summarize(res)
show = ['split', 'train_set', 'model', 'n_seeds', 'auroc_mean', 'auroc_std', 'f1_mean',
        'pd_at_pfa_0.01_mean', 'pd_at_pfa_0.05_mean', 'train_seconds_mean']
pd.set_option('display.width', 200)
summary[show].round(4)

,split,train_set,model,n_seeds,auroc_mean,auroc_std,f1_mean,pd_at_pfa_0.01_mean,pd_at_pfa_0.05_mean,train_seconds_mean
4,test,train,Random Forest,5,0.9617,0.0003,0.8970,0.5059,0.7821,0.2412
0,test,train,Gradient Boosting,5,0.9572,0.0000,0.9024,0.3500,0.7248,0.3464
5,test,train,SVM RBF,1,0.9386,NaN,0.8848,0.1068,0.5696,0.0438
2,test,train,"MLP (100, 50)",5,0.9343,0.0038,0.8794,0.1687,0.5319,0.3099
3,test,train,MLP small (10),5,0.9064,0.0079,0.8438,0.1601,0.4512,0.1926
1,test,train,Logistic Regression,1,0.8949,NaN,0.8383,0.1452,0.3296,0.0110
6,test,train_large,Gradient Boosting,5,0.9773,0.0012,0.9257,0.5628,0.8722,0.3734
10,test,train_large,Random Forest,5,0.9763,0.0003,0.9185,0.6721,0.8456,0.3868
8,test,train_large,"MLP (100, 50)",5,0.9751,0.0074,0.9231,0.5422,0.8612,1.4174
11,test,train_large,SVM RBF,1,0.9528,NaN,0.9027,0.1884,0.6996,0.5337


## 2. Riferimento con tutti i dati di training

Gradient Boosting addestrato sull'**intero** `train_dataset.csv` (52,7 M righe), compresso nei suoi ~3,3 M vettori unici con il numero di occorrenze come peso. È il riferimento "più dati possibile", paragonabile a `prova.py`. Il test di riferimento resta valido: i suoi vettori non compaiono mai nel file di training.

In [4]:
from sklearn.ensemble import HistGradientBoostingClassifier
DATA_DIR = Path(os.environ.get('QDETECT_DATA_DIR', 'output/split')).expanduser()
t0 = time.time()
parts = []
for ch in pd.read_csv(DATA_DIR / 'train_dataset.csv', usecols=qd.FEATURES + [qd.TARGET], chunksize=4_000_000):
    parts.append(ch.groupby(qd.FEATURES + [qd.TARGET]).size())
    if len(parts) >= 4:
        parts = [pd.concat(parts).groupby(level=list(range(9))).sum()]
full = pd.concat(parts).groupby(level=list(range(9))).sum().rename('w').reset_index()
print(f'{len(full):,} coppie (vettore, etichetta) da {int(full.w.sum()):,} righe in {time.time()-t0:.0f} s')

Xf = P.preprocessor.transform(full[qd.FEATURES].to_numpy(float))
yf = full[qd.TARGET].to_numpy(int)
t0 = time.time()
gb_full = HistGradientBoostingClassifier(random_state=42).fit(Xf, yf, sample_weight=full['w'].to_numpy(float))
t_fit = time.time() - t0
full_rows = []
for es, (Xe, ye) in {'test': (P.X_test, P.y_test), **P.extra}.items():
    full_rows.append(qd.evaluate('Gradient Boosting (train completo)', ye, gb_full.predict_proba(Xe)[:, 1],
                                 family='classical', seed=42, split=es, n_train=int(full['w'].sum()),
                                 train_seconds=t_fit, notes='train_set=full_file'))
qd.append_results(RESULTS, full_rows)
pd.DataFrame(full_rows)[['model', 'split', 'auroc', 'auroc_lo', 'auroc_hi', 'f1', 'pd_at_pfa_0.01', 'train_seconds']].round(4)

3,408,781 coppie (vettore, etichetta) da 52,752,000 righe in 16 s


,model,split,auroc,auroc_lo,auroc_hi,f1,pd_at_pfa_0.01,train_seconds
0,Gradient Boosting (train completo),test,0.9899,0.9879,0.9920,0.9549,0.7892,9.8267
1,Gradient Boosting (train completo),test_original,0.9996,0.9994,0.9998,0.9910,0.9924,9.8267


## 3. Test significativi (DeLong) sul test di riferimento

In [5]:
# Seed 42, modelli addestrati su 20.000 righe: rifitto per avere i punteggi.
scores = {}
for name, (build, _) in B.MODELS.items():
    m = build(42).fit(P.X_train_large, P.y_train_large)
    scores[name] = B.score(m, P.X_test)[0]
scores['Gradient Boosting (train completo)'] = gb_full.predict_proba(P.X_test)[:, 1]
ref = max(scores, key=lambda k: qd.auroc_ci(P.y_test, scores[k])[0])
rows = [{'confronto': f'{ref} vs {k}', **qd.delong_test(P.y_test, scores[ref], scores[k])}
        for k in scores if k != ref]
pd.DataFrame(rows).round(4)

,confronto,auroc_a,auroc_b,diff,z,p_value
0,Gradient Boosting (train completo) vs Logistic...,0.9899,0.9020,0.0879,21.0573,0.0
1,Gradient Boosting (train completo) vs MLP smal...,0.9899,0.9474,0.0425,15.0428,0.0
2,Gradient Boosting (train completo) vs MLP (100...,0.9899,0.9794,0.0105,8.3728,0.0
3,Gradient Boosting (train completo) vs SVM RBF,0.9899,0.9528,0.0371,13.7967,0.0
4,Gradient Boosting (train completo) vs Random F...,0.9899,0.9761,0.0138,12.5309,0.0
5,Gradient Boosting (train completo) vs Gradient...,0.9899,0.9756,0.0143,10.3406,0.0
